In [ ]:
from trouver.machine_learning.tokenize.def_and_notat_token_classification import _get_main_text_lines, _append_to_pieces_start_and_end, _divide_main_text, _html_tags_from_token_preds, _consolidate_token_preds, _collate_html_tags

In [ ]:
from collections.abc import Callable
from typing import Optional

import bs4
from transformers import pipelines

from trouver.helper.html import HTMLTagWithIndices

from trouver.obsidian.vault import VaultNote



## Use the trained model

See https://huggingface.co/docs/transformers/tasks/token_classification for training a token classification model.

In [ ]:
# Helper functions

In [ ]:
soup = bs4.BeautifulSoup('', 'html.parser')
tag = soup.new_tag('b', style="border-width:1px;border-style:solid;padding:3px", definition="")
tag.string = 'hi'
tag

<b definition="" style="border-width:1px;border-style:solid;padding:3px">hi</b>

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _ranges_overlap(
        current_1: HTMLTagWithIndices,
        current_2: HTMLTagWithIndices
        # current_1: tuple[bs4.element.Tag, int, int],
        # current_2: tuple[bs4.element.Tag, int, int]
        ) -> bool:
    """
    Based on https://stackoverflow.com/a/64745177

    Helper function to `_collate_html_tags`, `_consolidate_token_preds`.
    """
    return max(current_1.start, current_2.start) < min(current_1.end, current_2.end)

In [ ]:
#| hide
# In actuality, there should be bs4.element.Tag objects in place of ''.
assert _ranges_overlap(HTMLTagWithIndices('', 3, 8), HTMLTagWithIndices('', 6, 12))
assert _ranges_overlap(HTMLTagWithIndices('', 3, 8), HTMLTagWithIndices('', 3, 4))
assert not _ranges_overlap(HTMLTagWithIndices('', 3, 8), HTMLTagWithIndices('', 8, 9))
assert _ranges_overlap(HTMLTagWithIndices('', 6, 12), HTMLTagWithIndices('', 3, 8))

In [ ]:
# TODO: test

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _get_token_preds_by_dividing_main_text(
        main_text: str,
        pipeline: pipelines.token_classification.TokenClassificationPipeline, # The token classification pipeline that is used to predict whether tokens are part of definitions or notations introduced in the text. Here, the tokenizer of this pipeline is used to estimate how many tokens a piece of subtext will have.
        excessive_space_threshold: int, 
        note: Optional[VaultNote] = None,
        # ) -> list[tuple[bs4.element.Tag, int, int]]:  # Tag element, start, end, where main_text[start:end] needs to be replaced by the tag element.
        ) -> list[HTMLTagWithIndices]:  # Tag element, start, end, where main_text[start:end] needs to be replaced by the tag element.
    """
    Divide the `main_text` into not-too-long pieces to return HTML tag predictions

    Helper function for `_format_main_text_and_add_html_tag_data`.
    """
    pieces_start_and_end = _divide_main_text(main_text, pipeline)
    cumulative_html_tags_in_main = []
    for start_of_piece, end_of_piece in pieces_start_and_end:
        # text = main_text[start_of_piece:end_of_piece]
        text = main_text[start_of_piece:]
        html_tags_in_piece: list[HTMLTagWithIndices] = _html_tags_from_token_preds(
            text, pipeline(text), excessive_space_threshold, note)
        html_tags_in_piece = _consolidate_token_preds(
            text, html_tags_in_piece)
        # start and end indices need to be re-adjusted with respect to their places in `main_text`
        html_tags_for_piece_in_main_text: list[HTMLTagWithIndices] = [
            HTMLTagWithIndices(tag, start_of_piece + start, start_of_piece + end)
            for tag, start, end in html_tags_in_piece]
        cumulative_html_tags_in_main = _collate_html_tags(
            cumulative_html_tags_in_main, html_tags_for_piece_in_main_text)
    return cumulative_html_tags_in_main

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def get_def_and_notat_predictions(
        main_text: str,
        pipeline: pipelines.token_classification.TokenClassificationPipeline,
        excessive_space_threshold: int,
        note: Optional[VaultNote] = None
        ) -> list[HTMLTagWithIndices]:
    """
    Identifies definitions and notations in the text using the ML pipeline.

    Returns a list of `HTMLTagWithIndices` containing the predicted tags 
    and their start/end indices in `main_text`.
    """
    # 1. Divide text (if needed) and get raw token predictions
    html_tags_to_add = _get_token_preds_by_dividing_main_text(
        main_text, pipeline, excessive_space_threshold, note)
    
    # 2. Return the structured data directly
    # Note: _get_token_preds_by_dividing_main_text already returns 
    # list[HTMLTagWithIndices] after calling `consolidate_token_preds`.
    return html_tags_to_add

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
#| export machine_learning.tokenize.def_and_notat_token_classification
def mark_def_and_notat_predictions(
        main_text: str, # The original text.
        predictions: list[HTMLTagWithIndices], # The list of predictions (ranges and metadata) returned by `get_def_and_notat_predictions`. Assumes these are sorted and non-overlapping (which `get_def_and_notat_predictions` ensures).
        formatter: Callable[[str, HTMLTagWithIndices], str] # A function that takes the *extracted substring* (the text being marked) and the prediction object, and returns the *formatted string* to replace it with.
        ) -> str: # The modified text with predictions marked.
    """
    Applies formatting to the `main_text` based on the definition and notation predictions.
    """
    # Iterate backwards so that index changes don't affect subsequent replacements
    # (HTMLTagWithIndices are typically sorted by start index, so reversing is safe)
    sorted_preds = sorted(predictions, key=lambda x: x.start, reverse=True)
    
    formatted_text_list = list(main_text)
    
    for pred in sorted_preds:
        start, end = pred.start, pred.end
        original_substring = main_text[start:end]
        
        # Apply the user-defined formatting rule
        replacement_text = formatter(original_substring, pred)
        
        # Replace the slice in the text
        # (Using list slicing for efficiency vs repeated string concatenation)
        formatted_text_list[start:end] = list(replacement_text)
        
    return "".join(formatted_text_list)

In [ ]:
from bs4 import BeautifulSoup
# String: "The Galois group $\operatorname{Gal}(L/K)$ is..."
# Indices breakdown:
# "The Galois group " -> len 17 (indices 0-16)
# "$" -> index 17
# "\operatorname{Gal}(L/K)" -> len 23
# "$" -> index 17 + 1 + 23 = 41
# String slice [17:42] covers "$\operatorname{Gal}(L/K)$"
text = r"The Galois group $\operatorname{Gal}(L/K)$ is..."
soup = BeautifulSoup("", 'html.parser')

# Pred 1: Definition "Galois group" (indices 4-16)
tag_def = soup.new_tag("b", definition="")
pred_def = HTMLTagWithIndices(tag_def, 4, 16)

# Pred 2: Notation "$\operatorname{Gal}(L/K)$"
# We want to wrap the WHOLE math string including $.
# Start: 17
# End: 42 (17 + 25 chars)
tag_notat = soup.new_tag("span", notation="")
pred_notat = HTMLTagWithIndices(tag_notat, 17, 42)

predictions = [pred_def, pred_notat]

def bracket_formatter(subtext, pred):
    label = "DEF" if 'definition' in pred.tag.attrs else "NOT"
    return f"[{label}:{subtext}]"

result = mark_def_and_notat_predictions(text, predictions, bracket_formatter)

# Expected: "The [DEF:Galois group] [NOT:$\operatorname{Gal}(L/K)$] is..."
expected_str = r"The [DEF:Galois group] [NOT:$\operatorname{Gal}(L/K)$] is..."

assert result == expected_str, f"Test 1 Failed.\nExpected: {expected_str}\nGot:      {result}"
print("Test 1 Passed:", result)

# --- Test 2: HTML Formatter ---
# Checks that we can regenerate standard HTML tags
def simple_html_formatter(subtext, pred):
    # Important: Clone the tag if you want to be pure, 
    # but modifying pred.tag is usually fine for one-pass formatting.
    pred.tag.string = subtext
    return str(pred.tag)

result_html = mark_def_and_notat_predictions(text, predictions, simple_html_formatter)

# Expected: "The <b definition="">Galois group</b> <span notation="">$\operatorname{Gal}(L/K)$</span> is..."
expected_html_fragment_1 = r'<b definition="">Galois group</b>'
expected_html_fragment_2 = r'<span notation="">$\operatorname{Gal}(L/K)$</span>'

assert expected_html_fragment_1 in result_html
assert expected_html_fragment_2 in result_html
print("Test 2 Passed:", result_html)

Test 1 Passed: The [DEF:Galois group] [NOT:$\operatorname{Gal}(L/K)$] is...
Test 2 Passed: The <b definition="">Galois group</b> <span notation="">$\operatorname{Gal}(L/K)$</span> is...


In [ ]:
from unittest.mock import MagicMock

class MockTokenizer:
    """Mocks a HuggingFace tokenizer."""
    model_max_length = 512
    
    def __call__(self, text):
        # CORRECTED: Return a dict, not SimpleNamespace
        length = len(text) // 5 + 1
        return {'input_ids': [0] * length}

# Update the pipeline to use the corrected tokenizer
class MockPipeline:
    """Mocks the TokenClassificationPipeline."""
    def __init__(self, preds):
        self.tokenizer = MockTokenizer()
        self._preds = preds

    def __call__(self, text):
        return self._preds

In [ ]:

#| hide
from types import SimpleNamespace

# 1. Setup Mock Note
mock_note = SimpleNamespace(name="Test Note", path=lambda: "test/path/Test Note.md") 

# 2. Setup Input Text
# "The Galois group $\operatorname{Gal}(L/K)$ of the extension L/K is..."
# Indices:
# T=0, h=1, e=2,  =3
# G=4, a=5, l=6, o=7, i=8, s=9,  =10
# g=11, r=12, o=13, u=14, p=15
#  =16
# $=17, \=18, o=19, p=20 ...
text_input = r"The Galois group $\operatorname{Gal}(L/K)$ of the extension L/K is..."

# 3. Setup Mock Predictions
# We simulate the model finding:
# A. 'Galois group' as a definition.
# B. 'Gal' (inside the latex) as a notation.
mock_preds = [
    # Definition: "Galois group" (indices 4-16)
    {
        'entity': 'B-definition', 'score': 0.99, 'index': 1, 
        'word': 'Galois', 'start': 4, 'end': 10
    },
    {
        'entity': 'I-definition', 'score': 0.99, 'index': 2, 
        'word': 'group', 'start': 11, 'end': 16
    },
    # Notation: Model finds 'Gal' inside '$\operatorname{Gal}(L/K)$'
    # '$\operatorname{Gal}(L/K)$' starts at 17.
    # $ = 17
    # \operatorname{ = 18 to 31 (len 13)
    # Gal = 31 to 34
    {
        'entity': 'B-notation', 'score': 0.98, 'index': 3, 
        'word': 'Gal', 'start': 31, 'end': 34
    }
]

mock_pipeline = MockPipeline(mock_preds)

# 4. Run Function
preds = get_def_and_notat_predictions(
    main_text=text_input,
    pipeline=mock_pipeline,
    excessive_space_threshold=2,
    note=mock_note,
)

# 5. Assertions

# Verify we got 2 items
assert len(preds) == 2, f"Expected 2 predictions, got {len(preds)}"

# --- Check Definition ("Galois group") ---
# The function might return them in order of appearance. 
# 'Galois group' starts at 4, '$\operatorname...$' starts at 17.
def_pred = preds[0]
assert def_pred.tag.text.strip() == 'Galois group'
assert 'definition' in def_pred.tag.attrs
assert def_pred.start == 4
assert def_pred.end == 16

# --- Check Notation ("$\operatorname{Gal}(L/K)$") ---
# The model predicted 'Gal' (31-34).
# The logic should assume 'Gal' implies the surrounding math mode string is the notation.
# The full math string is `$\operatorname{Gal}(L/K)$`.
# Start index: 17 ($)
# End index: 17 + len('$\operatorname{Gal}(L/K)$') = 17 + 25 = 42
notat_pred = preds[1]
assert 'notation' in notat_pred.tag.attrs
assert notat_pred.start == 17
assert notat_pred.end == 42 # Check length of string manually if this fails
assert notat_pred.tag.text == r'$\operatorname{Gal}(L/K)$'

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def predict_and_mark_def_and_notats(
        main_text: str, # The text to run predictions on and format.
        pipeline: pipelines.token_classification.TokenClassificationPipeline, # The token classification pipeline.
        # note: VaultNote, # The note associated with the text (used for debugging/logging).
        formatter: Callable[[str, HTMLTagWithIndices], str], # The function that formats the text based on predictions.
        excessive_space_threshold: int # Threshold for detecting excessive spacing in predictions.
        ) -> str: # The formatted text.
    """
    Runs definition and notation detection on `main_text` and applies the `formatter`.
    """
    predictions = get_def_and_notat_predictions(
        main_text, pipeline, excessive_space_threshold)
    return mark_def_and_notat_predictions(main_text, predictions, formatter)

In [ ]:
from bs4 import BeautifulSoup
from unittest.mock import MagicMock
from fastcore.test import test_eq # Import fastcore's test_eq

# --- Setup ---
text_input = r"The Galois group $\operatorname{Gal}(L/K)$ is..."

mock_preds = [
    {'entity': 'B-definition', 'score': 0.99, 'index': 1, 'word': 'Galois', 'start': 4, 'end': 10},
    {'entity': 'I-definition', 'score': 0.99, 'index': 2, 'word': 'group', 'start': 11, 'end': 16},
    {'entity': 'B-notation', 'score': 0.98, 'index': 3, 'word': 'Gal', 'start': 31, 'end': 34}
]
mock_pipeline = MockPipeline(mock_preds)

# mock_note = MagicMock()
# mock_note.name = "Test Note"
# mock_note.path.return_value = "test_note.md"

def html_formatter(subtext, pred):
    pred.tag.string = subtext
    return str(pred.tag)

# --- Run Function ---
result_html = predict_and_mark_def_and_notats(
    main_text=text_input,
    pipeline=mock_pipeline,
    formatter=html_formatter,
    excessive_space_threshold=2
)

# --- Define Expectations ---
# Based on the output you saw earlier, we define the exact expected string.
# We include the specific style attributes that seem to be generated by default in your environment.
expected_html = (
    r'The <b definition="" style="border-width:1px;border-style:solid;padding:3px">Galois group</b> '
    r'<span notation="" style="border-width:1px;border-style:solid;padding:3px">$\operatorname{Gal}(L/K)$</span> is...'
)

# --- Assertion with fastcore ---
# test_eq will raise an error if they differ and print:
# "test_eq: expected vs actual" followed by the two strings.
# If they are long, it usually helps spot the difference.
test_eq(result_html, expected_html)

print("Test 1 (HTML) Passed with fastcore!")


# --- Test 2: Bracket Formatter (Debug/Sanity Check) ---
def bracket_formatter(subtext, pred):
    tag_type = "DEF" if 'definition' in pred.tag.attrs else "NOT"
    return f"[{tag_type}:{subtext}]"

result_bracket = predict_and_mark_def_and_notats(
    main_text=text_input,
    pipeline=mock_pipeline,
    # note=mock_note,
    formatter=bracket_formatter,
    excessive_space_threshold=2
)

expected_bracket = r"The [DEF:Galois group] [NOT:$\operatorname{Gal}(L/K)$] is..."
assert result_bracket == expected_bracket, f"Bracket test failed. Got: {result_bracket}"
print("Test 2 (Bracket) Passed!")

Test 1 (HTML) Passed with fastcore!
Test 2 (Bracket) Passed!
